In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# ✅ 전체 코드: Balanced Batch 학습용 (1번 + 4번 농가)


import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.preprocessing import StandardScaler
from torch import nn
# 1번과 4번 농가의 원본 데이터의 선택된 컬럼을 합쳐서 scaler를 fit 한다.
# 🔹 Step 1. 데이터 불러오기 및 정제
df_1 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/content/TS_Timeseries/tom1/토마토_환경데이터_1번농가(그린CS).csv', low_memory=False)
df_4 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/content/TS_Timeseries/tom4/토마토_환경데이터_4번농가(그린CS).csv', low_memory=False)

# 두 데이터셋에서 공통으로 존재하는 컬럼 추출
common_features = list(set(df_1.columns).intersection(set(df_4.columns)))

# 타겟 변수 및 피처 결정 ('date'는 제외)
target_col = "현재EC(dS)"
feature_cols = [col for col in common_features if col not in ['date', target_col]]

# 2. 수치형 피처만 선별 (문자열로 표현된 시간 등은 제외)
numeric_cols = []
for col in feature_cols:
    try:
        # 두 데이터셋 모두 숫자로 변환 가능한지 확인
        df_1[col].astype(float)
        df_4[col].astype(float)
        numeric_cols.append(col)
    except Exception as e:
        continue

# 타겟 변수도 수치형이라고 가정하고 결측값 제거
df_1 = df_1.dropna(subset=numeric_cols + [target_col, 'date'])
df_4 = df_4.dropna(subset=numeric_cols + [target_col, 'date'])

# 날짜 기준 정렬
df_1 = df_1.sort_values("date")
df_4 = df_4.sort_values("date")

# 3. 농가별 독립적인 정규화 (StandardScaler) 적용
# 피처 정규화
scaler_1 = StandardScaler()
scaler_4 = StandardScaler()

df_1[numeric_cols] = scaler_1.fit_transform(df_1[numeric_cols])
df_4[numeric_cols] = scaler_4.fit_transform(df_4[numeric_cols])

# 타겟 변수도 별도 스케일러를 사용
target_scaler_1 = StandardScaler()
target_scaler_4 = StandardScaler()

df_1[target_col] = target_scaler_1.fit_transform(df_1[[target_col]])
df_4[target_col] = target_scaler_4.fit_transform(df_4[[target_col]])

# 4. 시퀀스 데이터 생성
def create_sequences(df, seq_len=24):
    X, y = [], []
    for i in range(len(df) - seq_len):
        # seq_len 길이만큼의 피처 시퀀스 생성
        X.append(df[numeric_cols].iloc[i:i+seq_len].values)
        # 시퀀스 이후 시점의 타겟 값 선택
        y.append(df[target_col].iloc[i+seq_len])
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

X1, y1 = create_sequences(df_1)
X4, y4 = create_sequences(df_4)

# 5. 두 농가 데이터를 통합한 BalancedDataset 정의
class BalancedDataset(Dataset):
    def __init__(self, X1, y1, X2, y2):
        # 두 데이터셋을 연결 (각 농가별 스케일이 다르더라도, 모델은 정규화된 값을 학습)
        self.X = torch.cat([X1, X2])
        self.y = torch.cat([y1, y2])
        # 도메인 라벨 생성: 0은 1번 농가, 1은 4번 농가
        self.domains = torch.cat([torch.zeros(len(X1)), torch.ones(len(X2))])

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 6. Balanced Batch Sampler 정의 (각 도메인당 균등한 샘플 수를 보장)
class BalancedBatchSampler(Sampler):
    def __init__(self, domains):
        self.domains = domains
        self.indices = self._balance_indices()

    def _balance_indices(self):
        # numpy array로 변환 후 도메인별 인덱스 나눔
        labels = self.domains.numpy()
        idx_0 = np.where(labels == 0)[0]
        idx_1 = np.where(labels == 1)[0]
        min_len = min(len(idx_0), len(idx_1))
        np.random.shuffle(idx_0)
        np.random.shuffle(idx_1)
        balanced_indices = np.hstack([idx_0[:min_len], idx_1[:min_len]])
        np.random.shuffle(balanced_indices)
        return balanced_indices.tolist()

    def __iter__(self):
        return iter(self.indices)

    def __len__(self):
        return len(self.indices)

dataset = BalancedDataset(X1, y1, X4, y4)
sampler = BalancedBatchSampler(dataset.domains)
dataloader = DataLoader(dataset, batch_size=32, sampler=sampler)

# 7. Transformer 기반 시계열 모델 정의
class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim, model_dim=64, num_heads=4, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, model_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(model_dim, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, input_dim)
        x = self.input_proj(x)
        x = self.transformer(x)
        # 평균 풀링: sequence 길이에 대한 평균
        x = x.mean(dim=1)
        return self.fc(x).squeeze()

# 8. 모델 초기화 및 학습 루프 (Pretraining 단계)
model = TimeSeriesTransformer(input_dim=len(numeric_cols))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for xb, yb in dataloader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

KeyboardInterrupt: 

In [3]:
# 추가 학습이 끝나면 모델 저장
import torch
torch.save(model.state_dict(), "/content/drive/MyDrive/Colab Notebooks/Tomato_Timeseries/greenscs_finetuned1_1.pt")

NameError: name 'model' is not defined

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from torch import nn

# 기존 전처리 및 시퀀스 생성 코드는 그대로 사용한다고 가정
# (앞서 제공한 코드에서 df_1, numeric_cols, target_col, create_sequences 함수 등을 포함)

# hold-out 테스트: 1번 농가의 마지막 20% 데이터를 테스트셋으로 분리
test_ratio = 0.2
n_total = len(df_1)
n_test = int(test_ratio * n_total)
n_train = n_total - n_test

# 테스트 데이터는 df_1의 마지막 부분을 사용 (순서가 중요한 시계열 데이터를 고려)
df_test = df_1.iloc[n_train:].reset_index(drop=True)

# 시퀀스 길이는 학습과 동일하게 설정 (예: 24)
seq_len = 24
X_test, y_test = create_sequences(df_test, seq_len=seq_len)

# 테스트 데이터셋 생성
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 모델을 평가 모드로 전환
model.eval()
test_loss = 0.0
all_preds = []
all_targets = []

criterion = nn.MSELoss()

with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb)
        loss = criterion(preds, yb)
        test_loss += loss.item() * xb.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(yb.cpu().numpy())

# 전체 테스트 손실(MSE) 계산
test_loss /= len(test_dataset)
print(f"Test Loss (MSE): {test_loss:.4f}")

# 필요시, 예측 결과를 단일 배열로 변환 (역정규화가 필요하면 target_scaler_1.inverse_transform 사용)
all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

# 예시: 테스트 예측값과 실제값의 일부를 출력
for i in range(5):
    print(f"Prediction: {all_preds[i]:.4f}, Ground Truth: {all_targets[i]:.4f}")
